# Práctica 1

**Duración:** 10–15 minutos · **Entorno:** Google Colab con GPU recomendada

Cargarás un modelo pequeño, verás cómo el texto se convierte en tokens y generarás tu primera respuesta. Después probarás cómo cambia al modificar los parámetros de generación.

**Objetivos**
- Distinguir prompt, tokens de entrada y tokens generados.
- Ejecutar un modelo instruct con Transformers.
- Observar el efecto de `max_new_tokens` y `temperature`.
- Medir el tiempo de generación y los tokens por segundo.

> En Colab, selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. La primera carga descarga el modelo y puede tardar unos minutos.

Usaremos [SmolLM3-3B](https://huggingface.co/HuggingFaceTB/SmolLM3-3B), un modelo instruct compacto con soporte en multiples idiomas.

## 1. Preparar el entorno
Instalamos Transformers y Accelerate. PyTorch suele venir ya instalado en Colab.

Aquí comprobamos la versión de PyTorch y el procesador (hardware) que ejecutará el modelo.

In [1]:
%pip -q install -U "transformers>=4.53.0" accelerate

In [2]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"PyTorch: {torch.__version__}")
print(f"Dispositivo: {device_name}")
if not torch.cuda.is_available():
    print("Consejo: activa una GPU en Colab para que la generación sea más rápida.")
    print("En Colab, selecciona Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU")

PyTorch: 2.11.0+cu130
Dispositivo: Tesla T4


## 2. Cargar el modelo y el tokenizer
El tokenizer traduce texto a identificadores numéricos. El modelo recibe esos identificadores y predice tokens nuevos.

Esta fase deberá acabar con un "Modelo cargado".

In [3]:
MODEL_ID = "HuggingFaceTB/SmolLM3-3B"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=dtype,
)
model.eval()
print("Modelo cargado.")

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Modelo cargado.


## 3. Analizar y entender los tokens
Usamos la plantilla de chat del modelo para dar formato a la conversación. Fíjate en que un token no equivale necesariamente a una palabra.

* **messages** define una conversación con dos mensajes: system (instrucciones) y user (pregunta).
* **apply_chat_template()**: Convierte los mensajes al formato que entiende el modelo, añadiendo etiquetas como <|im_start|>system.
* **enable_thinking=False**: Solicita desactivar el razonamiento explícito del modelo.
* **tokenizer()**: Convierte el texto completo en tokens, representados mediante identificadores numéricos (input_ids).
* **.to(model.device)**: Mueve los tensores al dispositivo donde está cargado el modelo (CPU o GPU).
* **shape[1]**: Cuenta los tokens de entrada. En este caso, 76 tokens.

In [8]:
messages = [
    {"role": "system", "content": "Responde en español y de forma clara."},
    {"role": "user", "content": "¿Qué hace un tokenizer? Explícalo con un ejemplo breve."},
]

# 1. Aplicar la plantilla de conversación
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

# 2. Tokenizar
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 3. Obtener IDs y tokens legibles
token_ids = inputs["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(token_ids)

# 4. Mostrar resultados
print("=== PROMPT ===")
print(prompt)

print("\n=== TOKENS ===")
for i, (token, token_id) in enumerate(zip(tokens, token_ids)):
    print(f"{i:3} | {token_id:8} | {repr(token)}")

print(f"\nTotal de tokens: {len(token_ids)}")

=== PROMPT ===
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 02 November 2026
Reasoning Mode: /no_think

## Custom Instructions

Responde en español y de forma clara.

<|im_start|>user
¿Qué hace un tokenizer? Explícalo con un ejemplo breve.<|im_end|>
<|im_start|>assistant
<think>

</think>


=== TOKENS ===
  0 |   128011 | '<|im_start|>'
  1 |     9125 | 'system'
  2 |      198 | 'Ċ'
  3 |      567 | '##'
  4 |    34689 | 'ĠMetadata'
  5 |      271 | 'ĊĊ'
  6 |    81434 | 'Knowledge'
  7 |      356 | 'ĠC'
  8 |    28540 | 'utoff'
  9 |     2696 | 'ĠDate'
 10 |       25 | ':'
 11 |     5651 | 'ĠJune'
 12 |      220 | 'Ġ'
 13 |     2366 | '202'
 14 |       20 | '5'
 15 |      198 | 'Ċ'
 16 |    15724 | 'Today'
 17 |     2696 | 'ĠDate'
 18 |       25 | ':'
 19 |      220 | 'Ġ'
 20 |     2437 | '02'
 21 |     6841 | 'ĠNovember'
 22 |      220 | 'Ġ'
 23 |     2366 | '202'
 24 |       21 | '6'
 25 |      198 | 'Ċ'
 26 |    26197 | 'Reason'
 27 |      287 | 'ing

## 4. Generar una respuesta
`max_new_tokens` limita cuántos tokens nuevos puede generar el modelo. El primer llamado también puede incluir tiempo de inicialización; por eso calentamos el modelo antes de medir.

| Parámetro | Función |
|---|---|
| `max_new_tokens=100` | Máximo de 100 tokens generados. |
| `temperature=None` | Generación determinista mediante greedy decoding, por defecto si la configuración no lo sobrescribe. |
| `do_sample=False` | Selecciona el token más probable en cada paso. |
| `torch.inference_mode()` | Desactiva el cálculo de gradientes y reduce el consumo de memoria. |
| `torch.cuda.synchronize()` | Espera a que termine la GPU para medir correctamente el tiempo. |

In [9]:
def generate(messages, max_new_tokens=100, temperature=None):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    options = {"max_new_tokens": max_new_tokens, "do_sample": temperature is not None}
    if temperature is not None:
        options["temperature"] = temperature
    with torch.inference_mode():
        start = time.perf_counter()
        output = model.generate(**inputs, **options)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
    prompt_tokens = inputs["input_ids"].shape[1]
    new_tokens = output.shape[1] - prompt_tokens

    answer = tokenizer.decode(
        output[0][prompt_tokens:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return {
        "answer": answer,
        "input_tokens": prompt_tokens,
        "output_tokens": new_tokens,
        "seconds": elapsed,
        "tokens_per_second": new_tokens / elapsed if elapsed else 0, # Eso mide la velocidad media de generación, incluyendo el procesamiento inicial del prompt (prefill).
    }

# Calentamiento: no usamos esta ejecución para comparar tiempos.
_ = generate(messages, max_new_tokens=20)
result = generate(messages, max_new_tokens=100)
print(result["answer"])
print(f"\nEntrada: {result['input_tokens']} tokens")
print(f"Salida: {result['output_tokens']} tokens")
print(f"Tiempo de generación: {result['seconds']:.2f} s")
print(f"Velocidad: {result['tokens_per_second']:.2f} tokens/s")

Un tokenizer es una herramienta o función que divide un texto en unidades más pequeñas, generalmente palabras o tokens, para que puedan ser procesadas de manera más eficiente por un programa de computadora. Estos tokens pueden ser palabras, símbolos, números, etc.

**Ejemplo:**

Supongamos que tienes la siguiente oración: "El gato negro corre por el parque."

Un tokenizer dividiría esta oración en los siguientes tokens: ["

Entrada: 76 tokens
Salida: 100 tokens
Tiempo de generación: 6.48 s
Velocidad: 15.43 tokens/s


## 5. Experimentar con la generación
Prueba a cambiar `max_new_tokens` a 30 y luego a 150. ¿Qué cambia en la respuesta y en el tiempo?

Después compara dos temperaturas. Una temperatura baja suele producir respuestas más predecibles; una más alta, más variación. La temperatura solo tiene efecto cuando `do_sample=True`.

In [11]:
idea = [{"role": "user", "content": "Propón un nombre para una cafetería junto a una estación de tren y explica brevemente por qué."}]

for temp in (0.2, 0.7, 1.2):
    torch.manual_seed(7)
    result = generate(idea, max_new_tokens=70, temperature=temp)
    print(f"==> TEMPERATURA {temp} · {result['seconds']:.2f} s · {result['tokens_per_second']:.2f} tokens/s")
    print(result["answer"], "\n\n")

==> TEMPERATURA 0.2 · 3.56 s · 19.68 tokens/s
Nombre sugerido: "Tren y Café"

Explicación: "Tren y Café" es un nombre que combina dos elementos clave de la ubicación: la cercanía a la estación de tren y la oferta de café. Este nombre transmite la idea de que la cafetería es un lugar ideal para los viaj 


==> TEMPERATURA 0.7 · 3.12 s · 22.47 tokens/s
Proposición de nombre: "Parada de Café"

Explicación: "Parada de Café" es un nombre que refleja la ubicación de la cafetería junto a una estación de tren. La palabra "parada" evoca la idea de un lugar de descanso y espera, lo cual es común 


==> TEMPERATURA 1.2 · 3.04 s · 23.05 tokens/s
Proposición de nombre: *Trem Brevity*

Explicación: 

El nombre *Trem Brevity* (oraciones breves en inglés) se basa en la posibilidad de que muchos transeúntes se detengan a tomar un café brevemente durante sus desplazamientos en tren. 


